<a href="https://colab.research.google.com/github/dataguirre/curso-ia-ciencia-de-datos/blob/main/sesion6/groq_tools_and_skills.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Herramientas y Skills con la API de Groq — Lo básico

Este notebook enseña dos ideas que convierten un simple modelo de lenguaje en algo capaz de *actuar* y *especializarse*:

1. **Herramientas** (también conocidas como *function calling*) — permiten que el modelo llame funciones que tú defines, obtenga datos actualizados y ejecute trabajo de varios pasos.
2. **Skills** — empaquetan experiencia reutilizable (instrucciones + recursos) que se carga solo cuando es relevante.

Todo se ejecuta contra la **API de Groq**, que es compatible con OpenAI, así que las estructuras de solicitud/respuesta coinciden con el formato Chat Completions de OpenAI.

**Una nota sobre las "Skills":** las *Agent Skills* como funcionalidad gestionada y descubierta automáticamente son específicas de los productos Claude de Anthropic. Groq no tiene una funcionalidad de Skills integrada. Lo que sí es portable es el **patrón** — descubrimiento basado primero en metadatos y carga progresiva — y la Parte 2 reproduce ese patrón sobre el endpoint de chat de Groq para que puedas usarlo con cualquier modelo.

**Requisitos previos:** Python 3.9+, una clave de API de Groq obtenida en [console.groq.com/keys](https://console.groq.com/keys), configurada como la variable de entorno `GROQ_API_KEY`.

## 0. Setup

In [1]:
# Install the official Groq Python SDK
%pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.3 MB/s eta 0:00:00


In [2]:
import os
import json
from groq import Groq
from google.colab import userdata

# The client reads GROQ_API_KEY from the environment automatically.
# (You can also pass api_key="..." explicitly, but avoid hard-coding secrets.)
client = Groq(api_key=userdata.get("GROQ_KEY"))

# Default model. Groq serves several open models; swap freely:
#   "openai/gpt-oss-120b"                       -> strong general + tool use (used here)
#   "llama-3.3-70b-versatile"                   -> capable Llama option
#   "meta-llama/llama-4-scout-17b-16e-instruct" -> fast, lightweight
MODEL = "openai/gpt-oss-120b"

print("Client ready. Using model:", MODEL)

Client ready. Using model: openai/gpt-oss-120b


## 1. Herramientas (function calling)

El uso de herramientas es un **contrato** entre tu código y el modelo. Tú describes qué funciones existen y la forma de sus entradas; el modelo decide *cuándo* llamar una y *con qué argumentos*. El modelo nunca ejecuta nada por sí mismo — emite una solicitud estructurada, tu código la ejecuta y el resultado regresa a la conversación. Eso hace que el modelo se comporte menos como un generador de texto y más como una función que puedes invocar.

**Dónde se ejecuta realmente el código** importa, y hay tres categorías:

- **Herramientas locales (las ejecutas tú):** tú escribes el esquema *y* la implementación. El modelo devuelve una llamada a la herramienta; tu código la ejecuta y devuelve el resultado. Este es el grueso del uso real de herramientas — lógica de negocio personalizada, APIs internas, bases de datos.
- **Herramientas integradas / del lado del servidor (las ejecuta Groq):** para cosas como búsqueda web y ejecución de código, Groq ejecuta la herramienta en su propia infraestructura y te entrega la respuesta terminada. No tienes que orquestar un bucle.
- **Herramientas remotas / MCP (Groq orquesta proveedores externos):** Groq se conecta a un servidor MCP, descubre sus herramientas y ejecuta las llamadas del lado del servidor.

**Recurre a una herramienta cuando** la tarea requiera una acción con efectos secundarios (enviar un correo, escribir un registro), datos frescos/externos (los precios de hoy, una fila de una base de datos) o una forma de salida garantizada. **Omítela cuando** el modelo pueda responder con lo que ya sabe — resumir o traducir no necesita un ida y vuelta a una herramienta.

### 1.1 Define una herramienta y haz la primera llamada

La definición de una herramienta es un esquema JSON que describe la función: su `name`, una `description` clara (así es como el modelo decide cuándo usarla) y `parameters` tipados. Empezaremos con una función de clima simulada para que el notebook sea autónomo.

In [3]:
# 1) The implementation (a real one would call a weather API)
def get_weather(location: str, unit: str = "celsius") -> str:
    fake = {"Bogota": 19, "Tokyo": 26, "London": 14}
    temp = fake.get(location, 21)
    if unit == "fahrenheit":
        temp = round(temp * 9 / 5 + 32)
    return json.dumps({"location": location, "temperature": temp, "unit": unit, "condition": "clear"})

# 2) The schema the model sees
weather_tool = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the current weather for a city. Use when the user asks about weather or temperature.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "City name, e.g. 'Tokyo'"},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
            },
            "required": ["location"],
        },
    },
}

messages = [
    {"role": "system", "content": "You are a helpful assistant. Use tools when they help."},
    {"role": "user", "content": "What's the weather in Tokyo right now?"},
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=[weather_tool],
    tool_choice="auto",  # let the model decide
)

msg = response.choices[0].message
print("finish_reason:", response.choices[0].finish_reason)
print("tool_calls:", msg.tool_calls)

finish_reason: tool_calls
tool_calls: [ChatCompletionMessageToolCall(id='fc_bf41ce3c-9d6d-40a4-afb3-9fb938dcc278', function=Function(arguments='{"location":"Tokyo","unit":"celsius"}', name='get_weather'), type='function')]


Cuando el modelo quiere una herramienta, la respuesta regresa con `finish_reason == "tool_calls"` y una lista `tool_calls`. Cada llamada tiene:

- `id` — un id único que debes devolver junto con el resultado,
- `function.name` — qué función ejecutar,
- `function.arguments` — una **cadena JSON** de argumentos (la analizas tú mismo).

El modelo ha producido una *solicitud*, no una respuesta. Todavía no se ha ejecutado nada.

### 1.2 Cierra el ciclo: ejecuta y devuelve el resultado

Para obtener una respuesta en lenguaje natural, ejecutas la función, agregas un mensaje `tool` (con el `tool_call_id` correspondiente) y vuelves a llamar al modelo.

In [4]:
available = {"get_weather": get_weather}

# Add the assistant's tool-call message to the history
messages.append(msg)

# Execute each requested call and append its result
for tc in msg.tool_calls:
    fn = available[tc.function.name]
    args = json.loads(tc.function.arguments)
    result = fn(**args)
    messages.append({
        "role": "tool",
        "tool_call_id": tc.id,
        "name": tc.function.name,
        "content": result,
    })
print(messages)
# Ask again, now that the model can see the tool output
final = client.chat.completions.create(model=MODEL, messages=messages, tools=[weather_tool])
print(final.choices[0].message.content)

[{'role': 'system', 'content': 'You are a helpful assistant. Use tools when they help.'}, {'role': 'user', 'content': "What's the weather in Tokyo right now?"}, ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='User asks weather in Tokyo right now. Use function get_weather.', tool_calls=[ChatCompletionMessageToolCall(id='fc_bf41ce3c-9d6d-40a4-afb3-9fb938dcc278', function=Function(arguments='{"location":"Tokyo","unit":"celsius"}', name='get_weather'), type='function')]), {'role': 'tool', 'tool_call_id': 'fc_bf41ce3c-9d6d-40a4-afb3-9fb938dcc278', 'name': 'get_weather', 'content': '{"location": "Tokyo", "temperature": 26, "unit": "celsius", "condition": "clear"}'}]
The current weather in Tokyo is **26 °C** with clear skies.


### 1.3 El bucle agéntico (varios pasos)

Las tareas reales necesitan más de un ida y vuelta: el modelo puede llamar a una herramienta, observar el resultado y luego llamar a otra. El patrón canónico es un bucle `while` que sigue ejecutándose **mientras el modelo siga devolviendo llamadas a herramientas**, con un límite estricto para que no pueda girar indefinidamente.

In [5]:
import math

# A small toolbox
def calculate(expression: str) -> str:
    try:
        return json.dumps({"result": eval(expression, {"__builtins__": {}}, {"math": math})})
    except Exception as e:
        return json.dumps({"error": str(e)})

TOOLS = [
    weather_tool,
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate an arithmetic expression, e.g. '23 * 7 + 5'.",
            "parameters": {
                "type": "object",
                "properties": {"expression": {"type": "string", "description": "The expression to evaluate"}},
                "required": ["expression"],
            },
        },
    },
]
REGISTRY = {"get_weather": get_weather, "calculate": calculate}


def run_agent(user_query, max_iterations=8):
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use tools to get facts and do math."},
        {"role": "user", "content": user_query},
    ]
    resp = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, tool_choice="auto")

    for i in range(max_iterations):
        m = resp.choices[0].message
        if not m.tool_calls:
            return m.content  # model produced a final answer

        messages.append(m)
        print(f"Iteration {i + 1}: {len(m.tool_calls)} tool call(s)")
        for tc in m.tool_calls:
            args = json.loads(tc.function.arguments)
            print(f"   -> {tc.function.name}({args})")
            out = REGISTRY[tc.function.name](**args)
            print(f"      <- {out}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "name": tc.function.name, "content": out})

        resp = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, tool_choice="auto")

    return "Stopped: hit max_iterations."


print(run_agent("What's the temperature in Tokyo in fahrenheit, and what is that minus 10?"))

Iteration 1: 1 tool call(s)
   -> get_weather({'location': 'Tokyo', 'unit': 'fahrenheit'})
      <- {"location": "Tokyo", "temperature": 79, "unit": "fahrenheit", "condition": "clear"}
Iteration 2: 1 tool call(s)
   -> calculate({'expression': '79 - 10'})
      <- {"result": 69}
The current temperature in Tokyo is **79 °F**.  
Subtracting 10 degrees gives **69 °F**.


### 1.4 Dirigir el uso de herramientas

El parámetro `tool_choice` controla el comportamiento:

- `"auto"` (por defecto) — el modelo decide si llamar a una herramienta.
- `"required"` — fuerza al menos una llamada a herramienta (error 400 si se niega; bajar la `temperature` ayuda).
- `"none"` — prohíbe las herramientas en este turno.
- `{"type": "function", "function": {"name": "get_weather"}}` — fuerza una herramienta específica.

Otros hábitos útiles: mantén las **descripciones de las herramientas específicas** (el modelo se apoya por completo en ellas), haz que las herramientas devuelvan **JSON estructurado** en lugar de prosa, y mantén el conjunto de herramientas pequeño (3–5 es el punto ideal; deriva a un subconjunto cuando tengas muchas). Muchos modelos también admiten **llamadas a herramientas en paralelo** — varias `tool_calls` en un solo turno — así que ejecútalas todas antes de devolver los resultados.

## 2. Skills (el patrón portable)

Una **Skill** es experiencia reutilizable y bajo demanda: una carpeta que contiene instrucciones, metadatos y recursos opcionales (plantillas, documentos de referencia, scripts) que el modelo usa *automáticamente cuando es relevante*. A diferencia de un prompt — que es una guía puntual y específica de una conversación — una Skill se crea una vez y se reutiliza en muchas conversaciones.

La idea clave es la **divulgación progresiva**: cargar la información por etapas para que no esté toda en el contexto al mismo tiempo.

| Nivel | Cuándo se carga | Costo | Contenido |
|-------|-----------------|-------|-----------|
| **1 — Metadatos** | Siempre | mínimo (~nombre + descripción) | `name` y `description`: qué hace la skill y cuándo usarla |
| **2 — Instrucciones** | Cuando se activa la skill | bajo | el cuerpo de `SKILL.md`: el flujo de trabajo/guía en sí |
| **3 — Recursos y código** | Según se necesite | solo lo que se usa | archivos adicionales (plantillas, referencias) y scripts |

Anthropic ofrece esto como una funcionalidad gestionada; a continuación reproducimos el mecanismo en Groq para que veas exactamente cómo funciona. La unidad de una skill es un archivo `SKILL.md` con frontmatter en YAML (`name`, `description`) más un cuerpo en markdown.

### 2.1 Crea un par de Skills en disco

Crearemos dos skills pequeñas. Cada una es una carpeta con un `SKILL.md`; una además incluye un recurso de Nivel 3 (un archivo de plantilla).

In [6]:
import pathlib, textwrap

ROOT = pathlib.Path("skills")
ROOT.mkdir(exist_ok=True)

# Skill A: meeting minutes (with a bundled template = Level 3 resource)
(ROOT / "meeting-minutes").mkdir(exist_ok=True)
(ROOT / "meeting-minutes" / "SKILL.md").write_text(textwrap.dedent("""\
    ---
    name: meeting-minutes
    description: Turn raw meeting notes into clean minutes. Use when the user pastes meeting notes or asks for minutes, a recap, or action items.
    ---

    # Meeting Minutes

    Convert messy notes into structured minutes. Always produce these sections,
    in this order, and omit a section only if it is truly empty:

    1. **Summary** - 2 sentences max.
    2. **Decisions** - bullet list, each a single decision.
    3. **Action items** - bullet list as `- [owner] task (due: date or 'n/a')`.
    4. **Open questions** - bullet list.

    Keep it terse. Do not invent owners or dates that are not in the notes.
    A reusable skeleton lives in template.md.
    """))
(ROOT / "meeting-minutes" / "template.md").write_text(textwrap.dedent("""\
    ## Summary
    ...

    ## Decisions
    -

    ## Action items
    - [owner] task (due: n/a)

    ## Open questions
    -
    """))

# Skill B: release notes
(ROOT / "release-notes").mkdir(exist_ok=True)
(ROOT / "release-notes" / "SKILL.md").write_text(textwrap.dedent("""\
    ---
    name: release-notes
    description: Draft user-facing release notes from a changelog or list of merged changes. Use when the user provides commits, PR titles, or a changelog.
    ---

    # Release Notes

    Group changes under **Added**, **Changed**, **Fixed**, **Deprecated**.
    Write each line for end users (benefit-first), not for engineers.
    Lead with a one-line highlight. Skip internal-only refactors.
    """))

print("Created skills:")
for p in sorted(ROOT.glob("*/SKILL.md")):
    print(" -", p.parent.name)

Created skills:
 - meeting-minutes
 - release-notes


### 2.2 Nivel 1 — carga solo los metadatos

Al iniciar, analizas únicamente el frontmatter de cada skill. Esto es económico, así que puedes registrar muchas skills; el modelo solo se entera de que cada una *existe* y *cuándo usarla*.

In [7]:
def parse_frontmatter(skill_md_text):
    """Tiny YAML-frontmatter reader: returns (meta_dict, body_text)."""
    meta, body = {}, skill_md_text
    if skill_md_text.startswith("---"):
        _, fm, body = skill_md_text.split("---", 2)
        for line in fm.strip().splitlines():
            if ":" in line:
                k, v = line.split(":", 1)
                meta[k.strip()] = v.strip()
    return meta, body.strip()


def load_skill_registry(root="skills"):
    registry = {}
    for skill_md in pathlib.Path(root).glob("*/SKILL.md"):
        meta, _ = parse_frontmatter(skill_md.read_text())
        registry[meta["name"]] = {"description": meta["description"], "path": skill_md}
    return registry


registry = load_skill_registry()
for name, info in registry.items():
    print(f"{name}: {info['description']}")

meeting-minutes: Turn raw meeting notes into clean minutes. Use when the user pastes meeting notes or asks for minutes, a recap, or action items.
release-notes: Draft user-facing release notes from a changelog or list of merged changes. Use when the user provides commits, PR titles, or a changelog.


### 2.3 Niveles 2 y 3 — activa y luego carga las instrucciones (y los recursos)

Cuando llega una solicitud, seleccionamos la skill correspondiente a partir de los metadatos, leemos el cuerpo de su `SKILL.md` dentro del prompt de sistema y solo entonces, de forma opcional, incorporamos los recursos incluidos a los que hace referencia. Las instrucciones completas entran en el contexto **solo cuando se necesitan**.

In [8]:
def choose_skill(user_query, registry):
    """Ask the model which skill (if any) applies, using only Level-1 metadata."""
    catalog = "\n".join(f"- {n}: {i['description']}" for n, i in registry.items())
    routing = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content":
                "Pick the single best skill for the user's request, or 'none'. "
                "Reply with ONLY the skill name.\n\nSkills:\n" + catalog},
            {"role": "user", "content": user_query},
        ],
    )
    pick = routing.choices[0].message.content.strip().strip("`").lower()
    return pick if pick in registry else None


def run_with_skills(user_query, registry, load_resources=True):
    name = choose_skill(user_query, registry)
    system = "You are a helpful assistant."

    if name:  # Level 2: load the instructions
        meta, body = parse_frontmatter(registry[name]["path"].read_text())
        system += f"\n\nApply this skill:\n\n{body}"
        print(f"[triggered skill: {name}]")

        if load_resources:  # Level 3: pull in referenced files only if mentioned
            skill_dir = registry[name]["path"].parent
            for res in skill_dir.glob("*.md"):
                if res.name != "SKILL.md" and res.name in body:
                    system += f"\n\n--- {res.name} ---\n{res.read_text()}"
                    print(f"[loaded resource: {res.name}]")
    else:
        print("[no skill matched; answering directly]")

    answer = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user_query}],
    )
    return answer.choices[0].message.content


notes = """Standup 6/9. Ana: API migration done, deploying Thurs. Sam blocked on
auth keys - needs them from IT. Decided to drop the legacy export. Should we
support CSV too? Unresolved."""

print(run_with_skills("Turn these into minutes:\n" + notes, registry))

[triggered skill: meeting-minutes]
[loaded resource: template.md]
## Summary
API migration is complete and will be deployed on Thursday. Sam is blocked awaiting auth keys from IT, and the team decided to drop the legacy export.

## Decisions
- Drop the legacy export.

## Action items
- [Ana] Deploy the API migration (due: Thursday)  
- [Sam] Obtain auth keys from IT (due: n/a)

## Open questions
- Should we support CSV too?


Prueba la otra skill para ver cómo el enrutador elige una diferente:

In [9]:
changelog = """- merged: add dark mode toggle
- merged: fix crash when opening empty project
- merged: deprecate v1 /search endpoint
- merged: internal: rename util module"""

print(run_with_skills("Draft release notes from:\n" + changelog, registry))

[triggered skill: release-notes]
**Highlight:** Dark mode is now available, and we’ve squashed a crash and updated the search API for a smoother, more reliable experience.

### Added
- **Dark mode toggle** – Switch the interface to a dark theme anytime for easier on‑the‑eyes working in low‑light environments.

### Fixed
- **Crash on empty projects** – Opening a new, empty project no longer forces the app to close, keeping your workflow uninterrupted.

### Deprecated
- **v1 /search endpoint** – The old search API is being retired; please transition to the new version to continue enjoying fast, accurate search results.


### 2.4 Herramientas y Skills juntas

Se combinan de forma natural. Una **skill** aporta el *cómo* (el flujo de trabajo, el formato y el criterio), mientras que las **herramientas** aportan las *acciones y los datos*. Una skill de "triaje de soporte", por ejemplo, podría indicarle al modelo que llame a una herramienta `lookup_order` y luego dé formato al resultado de una manera específica. En la práctica, pasarías tanto las instrucciones de la skill activada (prompt de sistema) **como** el arreglo `tools` relevante al mismo bucle agéntico de la Parte 1.3.

## 3. Herramientas vs Skills vs Prompts

| | Qué es | Dónde vive | Ideal para |
|---|---|---|---|
| **Prompt** | Instrucciones puntuales para una sola conversación | En el mensaje | Dirección rápida y ad-hoc |
| **Herramienta** | Una función invocable con un esquema tipado | Tu código / los servidores de Groq | Acciones, datos frescos, forma de salida garantizada |
| **Skill** | Experiencia reutilizable cargada bajo demanda | Una carpeta `SKILL.md` | Flujos de trabajo consistentes y conocimiento de dominio en muchas conversaciones |

**Seguridad:** trata las skills como si instalaras software. Usa únicamente skills de fuentes en las que confíes, y audita cada archivo incluido (instrucciones *y* scripts) antes de ejecutarlos — una skill maliciosa puede dirigir al modelo para que haga mal uso de cualquier herramienta y dato a su alcance. Ten especial cuidado con las skills que descargan contenido desde URLs externas.

## Resumen y lecturas adicionales

- Las **herramientas** son un contrato: el modelo emite una llamada estructurada, tú (o Groq) la ejecutas y el resultado regresa. Impulsa el trabajo de varios pasos con un bucle `while` basado en si el modelo sigue devolviendo `tool_calls`.
- Las **herramientas integradas** (Groq Compound) se ejecutan del lado del servidor — cambia el modelo y omite el bucle.
- Las **Skills** empaquetan experiencia reutilizable y la cargan de forma progresiva: metadatos económicos siempre, instrucciones completas solo cuando se activan, recursos solo cuando se usan. Las Skills gestionadas son una funcionalidad de Anthropic; el *patrón* funciona en cualquier API de chat, como se muestra aquí con Groq.

Documentación:
- Uso de herramientas en Groq: https://console.groq.com/docs/tool-use/overview
- Llamado a herramientas locales en Groq: https://console.groq.com/docs/tool-use/local-tool-calling
- Herramientas integradas / Compound de Groq: https://console.groq.com/docs/compound
- Modelos de Groq: https://console.groq.com/docs/models
- Anthropic Agent Skills (la funcionalidad gestionada y el concepto): https://platform.claude.com/docs/en/agents-and-tools/agent-skills/overview
- Agent Skills (el estándar abierto y la especificación): https://agentskills.io